# Tesla Sales & Price — End-to-End ML Pipeline
Dataset: Tesla Deliveries 2015–2025 (sales volumes, pricing, specs, regional data)

## 0. Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# styling — dark bg looks nicer for EDA
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')
pd.set_option('display.max_columns', 20)

DATA_PATH = r'C:\Users\ASUS\Downloads\archive\tesla_deliveries_dataset_2015_2025.csv'
df = pd.read_csv(DATA_PATH)
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} cols")
df.head()

## 1. Data Preprocessing

In [ ]:
# quick sanity check
print(df.info())
print("\n--- Nulls ---")
print(df.isnull().sum())
print("\n--- Dupes ---")
print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
# build a proper datetime column so we can do time-series stuff later
df['Date'] = pd.to_datetime(df['Year'].astype(str) + '-' + df['Month'].astype(str) + '-01')

# sort chronologically — important for time series
df.sort_values('Date', inplace=True)
df.reset_index(drop=True, inplace=True)

# check dtypes after adding Date
df.dtypes

In [ ]:
# check for outliers in numeric cols using IQR
num_cols = ['Estimated_Deliveries', 'Production_Units', 'Avg_Price_USD',
            'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons', 'Charging_Stations']

def flag_outliers_iqr(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((series < q1 - k * iqr) | (series > q3 + k * iqr)).sum()

outlier_counts = {c: flag_outliers_iqr(df[c]) for c in num_cols}
print("Outlier counts (IQR method):")
for col, cnt in outlier_counts.items():
    print(f"  {col}: {cnt}")

# not removing outliers — they're legitimate high/low quarters, not data errors

In [ ]:
# encode categoricals
print("Unique regions:", df['Region'].nunique())
print("Unique models:", df['Model'].nunique())
print("Source types:", df['Source_Type'].unique())

# label encode for modeling later, keep originals for EDA
from sklearn.preprocessing import LabelEncoder

le_region = LabelEncoder()
le_model = LabelEncoder()
le_source = LabelEncoder()

df['Region_enc'] = le_region.fit_transform(df['Region'])
df['Model_enc'] = le_model.fit_transform(df['Model'])
df['Source_enc'] = le_source.fit_transform(df['Source_Type'])

print("\nRegion mapping:", dict(zip(le_region.classes_, le_region.transform(le_region.classes_))))
print("Model mapping:", dict(zip(le_model.classes_, le_model.transform(le_model.classes_))))

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# distributions of key numeric features
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
plot_cols = ['Estimated_Deliveries', 'Avg_Price_USD', 'Production_Units',
             'Range_km', 'CO2_Saved_tons', 'Charging_Stations']

for ax, col in zip(axes.ravel(), plot_cols):
    ax.hist(df[col], bins=40, edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=11)
    ax.axvline(df[col].median(), color='red', ls='--', lw=1.2, label='median')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# deliveries over time — aggregated monthly across all models/regions
monthly_total = df.groupby('Date')['Estimated_Deliveries'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_total['Date'], monthly_total['Estimated_Deliveries'], lw=1.5, marker='o', markersize=3)
ax.set_title('Total Monthly Deliveries (All Models & Regions)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Deliveries')
plt.tight_layout()
plt.show()

In [ ]:
# avg price trend per model
fig, ax = plt.subplots(figsize=(14, 5))
for model in df['Model'].unique():
    subset = df[df['Model'] == model].groupby('Date')['Avg_Price_USD'].mean()
    ax.plot(subset.index, subset.values, label=model, lw=1.3)

ax.set_title('Average Price Over Time by Model', fontsize=13)
ax.legend()
ax.set_ylabel('Avg Price (USD)')
plt.tight_layout()
plt.show()

In [ ]:
# deliveries by region — boxplot
fig, ax = plt.subplots(figsize=(10, 5))
df.boxplot(column='Estimated_Deliveries', by='Region', ax=ax)
ax.set_title('Deliveries by Region', fontsize=13)
plt.suptitle('')  # kill the auto title pandas adds
ax.set_ylabel('Deliveries')
plt.tight_layout()
plt.show()

In [ ]:
# correlation heatmap — numerics only
corr_cols = num_cols + ['Year', 'Month']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# deliveries by model — quick bar chart
model_totals = df.groupby('Model')['Estimated_Deliveries'].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
model_totals.plot(kind='bar', ax=ax, edgecolor='black')
ax.set_title('Total Deliveries by Model', fontsize=13)
ax.set_ylabel('Total Deliveries')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# time-based features
df['Quarter'] = df['Date'].dt.quarter
df['Half'] = (df['Month'] > 6).astype(int)  # H1=0, H2=1
df['YearMonth'] = df['Year'] * 100 + df['Month']

# production efficiency — ratio of deliveries to production
# values > 1 mean backlog clearing, < 1 means inventory buildup
df['Delivery_Efficiency'] = df['Estimated_Deliveries'] / df['Production_Units']

# price per km of range — kind of a "value" metric
df['Price_per_km'] = df['Avg_Price_USD'] / df['Range_km']

# price per kWh — another value metric buyers care about
df['Price_per_kWh'] = df['Avg_Price_USD'] / df['Battery_Capacity_kWh']

# CO2 efficiency per delivery
df['CO2_per_delivery'] = df['CO2_Saved_tons'] / df['Estimated_Deliveries']

# charging infra density — stations relative to deliveries in that row
df['Stations_per_1k_deliveries'] = (df['Charging_Stations'] / df['Estimated_Deliveries']) * 1000

# lag features — prev month deliveries (within same model-region group)
df.sort_values(['Model', 'Region', 'Date'], inplace=True)
df['Deliveries_lag1'] = df.groupby(['Model', 'Region'])['Estimated_Deliveries'].shift(1)
df['Deliveries_lag3'] = df.groupby(['Model', 'Region'])['Estimated_Deliveries'].shift(3)

# rolling 3-month avg deliveries
df['Deliveries_roll3'] = df.groupby(['Model', 'Region'])['Estimated_Deliveries'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

# year-over-year growth — same month last year
df['Deliveries_lag12'] = df.groupby(['Model', 'Region'])['Estimated_Deliveries'].shift(12)
df['YoY_growth'] = (df['Estimated_Deliveries'] - df['Deliveries_lag12']) / df['Deliveries_lag12']

# drop rows where lag features are NaN (first few months per group)
df_model = df.dropna(subset=['Deliveries_lag1', 'Deliveries_lag3']).copy()
print(f"Rows after dropping lag NaNs: {len(df_model)} (was {len(df)})")

# quick look at new features
df_model[['Date', 'Model', 'Region', 'Estimated_Deliveries', 'Delivery_Efficiency',
          'Price_per_km', 'Deliveries_lag1', 'Deliveries_roll3', 'YoY_growth']].head(10)

## 4. Regression Modeling
Predicting `Estimated_Deliveries` using price, specs, and engineered features.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# features for regression
feature_cols = [
    'Year', 'Month', 'Quarter',
    'Avg_Price_USD', 'Production_Units',
    'Battery_Capacity_kWh', 'Range_km', 'Charging_Stations',
    'Region_enc', 'Model_enc',
    'Delivery_Efficiency', 'Price_per_km', 'Price_per_kWh',
    'Deliveries_lag1', 'Deliveries_lag3', 'Deliveries_roll3',
    'Stations_per_1k_deliveries'
]

target = 'Estimated_Deliveries'

X = df_model[feature_cols].copy()
y = df_model[target].copy()

# replace any inf values from division features
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(X.median(), inplace=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# scale features — tree models don't care but linear ones do
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

In [ ]:
# throw a few models at it and see what sticks
models = {
    'Linear Regression': LinearRegression(),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
    'Lasso (α=0.5)': Lasso(alpha=0.5),
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42),
}

results = {}
for name, model in models.items():
    # linear models get scaled data, tree models get raw
    if 'Forest' in name or 'Boosting' in name:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
    else:
        model.fit(X_train_sc, y_train)
        preds = model.predict(X_test_sc)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R²': r2}
    print(f"{name:25s} | MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | R²: {r2:.4f}")

results_df = pd.DataFrame(results).T.sort_values('R²', ascending=False)
results_df

In [ ]:
# compare models visually
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric in zip(axes, ['MAE', 'RMSE', 'R²']):
    results_df[metric].plot(kind='barh', ax=ax, edgecolor='black')
    ax.set_title(metric, fontsize=12)
    ax.set_xlabel(metric)

plt.suptitle('Model Comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# feature importance from the best tree model
best_tree = models['Gradient Boosting']  # usually wins
importances = pd.Series(best_tree.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 7))
importances.plot(kind='barh', ax=ax, edgecolor='black')
ax.set_title('Feature Importance (Gradient Boosting)', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# actual vs predicted scatter for best model
best_preds = best_tree.predict(X_test)
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test, best_preds, alpha=0.4, s=20, edgecolors='k', linewidths=0.3)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect fit')
ax.set_xlabel('Actual Deliveries')
ax.set_ylabel('Predicted Deliveries')
ax.set_title('Actual vs Predicted (Gradient Boosting)')
ax.legend()
plt.tight_layout()
plt.show()

# residuals
residuals = y_test - best_preds
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(residuals, bins=40, edgecolor='black', alpha=0.7)
axes[0].set_title('Residual Distribution')
axes[0].axvline(0, color='red', ls='--')

axes[1].scatter(best_preds, residuals, alpha=0.4, s=20)
axes[1].axhline(0, color='red', ls='--')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residuals vs Predicted')
plt.tight_layout()
plt.show()

## 5. Hyperparameter Tuning
Fine-tuning Gradient Boosting with RandomizedSearchCV (faster than grid for this many params).

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 3, 5],
    'max_features': ['sqrt', 'log2', None],
}

gb_search = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=50,  # 50 random combos — decent coverage without taking forever
    cv=5,
    scoring='neg_mean_absolute_error',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
gb_search.fit(X_train, y_train)

print(f"\nBest MAE (CV): {-gb_search.best_score_:,.0f}")
print(f"Best params: {gb_search.best_params_}")

In [ ]:
# evaluate tuned model on test set
tuned_gb = gb_search.best_estimator_
tuned_preds = tuned_gb.predict(X_test)

print("--- Tuned Gradient Boosting ---")
print(f"MAE:  {mean_absolute_error(y_test, tuned_preds):,.0f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, tuned_preds)):,.0f}")
print(f"R²:   {r2_score(y_test, tuned_preds):.4f}")

# compare before vs after tuning
print(f"\n--- Before tuning ---")
print(f"MAE:  {results['Gradient Boosting']['MAE']:,.0f}")
print(f"R²:   {results['Gradient Boosting']['R²']:.4f}")

In [ ]:
# cross-val on tuned model — make sure we're not overfitting
cv_scores = cross_val_score(tuned_gb, X, y, cv=5, scoring='neg_mean_absolute_error')
print(f"5-fold CV MAE: {-cv_scores.mean():,.0f} ± {cv_scores.std():,.0f}")

## 6. Time Series Forecasting
Forecasting total monthly deliveries with ARIMA/SARIMAX + Prophet-style decomposition.

In [ ]:
# aggregate to monthly totals across all models & regions
ts = df.groupby('Date').agg(
    Total_Deliveries=('Estimated_Deliveries', 'sum'),
    Avg_Price=('Avg_Price_USD', 'mean'),
    Total_Production=('Production_Units', 'sum'),
).reset_index()

ts.set_index('Date', inplace=True)
ts = ts.asfreq('MS')  # month start frequency

print(f"Time series: {ts.index.min()} to {ts.index.max()}, {len(ts)} months")
ts.head()

In [ ]:
# decompose to see trend + seasonality
from statsmodels.tsa.seasonal import seasonal_decompose

decomp = seasonal_decompose(ts['Total_Deliveries'], model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title='Observed')
decomp.trend.plot(ax=axes[1], title='Trend')
decomp.seasonal.plot(ax=axes[2], title='Seasonal')
decomp.resid.plot(ax=axes[3], title='Residual')
plt.tight_layout()
plt.show()

In [ ]:
# stationarity check — ADF test
from statsmodels.tsa.stattools import adfuller

adf_result = adfuller(ts['Total_Deliveries'].dropna())
print(f"ADF statistic: {adf_result[0]:.4f}")
print(f"p-value: {adf_result[1]:.4f}")
print("Stationary" if adf_result[1] < 0.05 else "Non-stationary — will need differencing")

In [ ]:
# ACF/PACF plots to figure out ARIMA orders
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(ts['Total_Deliveries'].diff().dropna(), lags=30, ax=axes[0])
plot_pacf(ts['Total_Deliveries'].diff().dropna(), lags=30, ax=axes[1], method='ywm')
axes[0].set_title('ACF (differenced)')
axes[1].set_title('PACF (differenced)')
plt.tight_layout()
plt.show()

In [ ]:
# SARIMAX — seasonal ARIMA with exogenous variable (avg price)
from statsmodels.tsa.statespace.sarimax import SARIMAX

# train/test split — hold out last 12 months
train_ts = ts.iloc[:-12]
test_ts = ts.iloc[-12:]

# fitting SARIMAX(1,1,1)(1,1,1,12) — standard starting point
# using avg price as exogenous regressor
model_sarima = SARIMAX(
    train_ts['Total_Deliveries'],
    exog=train_ts[['Avg_Price']],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
)

sarima_fit = model_sarima.fit(disp=False, maxiter=500)
print(sarima_fit.summary().tables[0])
print(sarima_fit.summary().tables[1])

In [ ]:
# forecast next 12 months (test period)
forecast = sarima_fit.forecast(steps=12, exog=test_ts[['Avg_Price']])

# plot it
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train_ts.index, train_ts['Total_Deliveries'], label='Train', lw=1.5)
ax.plot(test_ts.index, test_ts['Total_Deliveries'], label='Actual (Test)', lw=2, color='green')
ax.plot(test_ts.index, forecast.values, label='SARIMAX Forecast', lw=2, ls='--', color='red')
ax.fill_between(test_ts.index, forecast.values * 0.9, forecast.values * 1.1, alpha=0.15, color='red')
ax.set_title('SARIMAX Forecast vs Actuals (Last 12 Months)', fontsize=13)
ax.legend()
ax.set_ylabel('Total Deliveries')
plt.tight_layout()
plt.show()

# forecast accuracy
ts_mae = mean_absolute_error(test_ts['Total_Deliveries'], forecast)
ts_rmse = np.sqrt(mean_squared_error(test_ts['Total_Deliveries'], forecast))
mape = np.mean(np.abs((test_ts['Total_Deliveries'] - forecast) / test_ts['Total_Deliveries'])) * 100
print(f"Forecast MAE:  {ts_mae:,.0f}")
print(f"Forecast RMSE: {ts_rmse:,.0f}")
print(f"MAPE:          {mape:.1f}%")

In [ ]:
# also try a simple exponential smoothing as baseline comparison
from statsmodels.tsa.holtwinters import ExponentialSmoothing

hw_model = ExponentialSmoothing(
    train_ts['Total_Deliveries'],
    trend='add',
    seasonal='add',
    seasonal_periods=12,
).fit(optimized=True)

hw_forecast = hw_model.forecast(12)

hw_mae = mean_absolute_error(test_ts['Total_Deliveries'], hw_forecast)
hw_mape = np.mean(np.abs((test_ts['Total_Deliveries'] - hw_forecast) / test_ts['Total_Deliveries'])) * 100
print(f"Holt-Winters MAE:  {hw_mae:,.0f}")
print(f"Holt-Winters MAPE: {hw_mape:.1f}%")
print(f"\nSARIMAX MAE:       {ts_mae:,.0f}")
print(f"SARIMAX MAPE:      {mape:.1f}%")

In [ ]:
# final comparison plot — both forecasts
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train_ts.index[-36:], train_ts['Total_Deliveries'].iloc[-36:], label='Train (last 3 yrs)', lw=1.5)
ax.plot(test_ts.index, test_ts['Total_Deliveries'], label='Actual', lw=2.5, color='green', marker='o')
ax.plot(test_ts.index, forecast.values, label=f'SARIMAX (MAPE={mape:.1f}%)', lw=2, ls='--', color='red')
ax.plot(test_ts.index, hw_forecast.values, label=f'Holt-Winters (MAPE={hw_mape:.1f}%)', lw=2, ls='--', color='orange')
ax.set_title('Forecast Comparison — SARIMAX vs Holt-Winters', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylabel('Total Deliveries')
plt.tight_layout()
plt.show()

## Summary

| Step | Key Result |
|------|-----------|
| **Preprocessing** | No nulls, built datetime index, encoded categoricals |
| **EDA** | Deliveries trending up, strong correlation between production & deliveries |
| **Feature Engineering** | 12+ new features — lags, rolling avgs, efficiency ratios, YoY growth |
| **Regression** | Gradient Boosting best R² among 5 models tested |
| **Hyperparameter Tuning** | RandomizedSearchCV (50 iters, 5-fold CV) on GB |
| **Time Series** | SARIMAX with price exog vs Holt-Winters baseline |